In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 179.2 MB/s eta 0:00:00


In [2]:
%%time
%%capture
!uv pip install transformers --system

CPU times: user 2.88 ms, sys: 2.11 ms, total: 4.99 ms
Wall time: 1.25 s


In [3]:
# from transformers import BartForConditionalGeneration, BartTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, MBartForConditionalGeneration
import torch

In [4]:
model_name = "BramVanroy/mbart-large-cc25-ft-amr30-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/540 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [6]:
model.config

MBartConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": true,
  "architectures": [
    "MBartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 250181,
  "dropout": 0.2,
  "dtype": "float32",
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_new_tokens": 512,
  "max_position_embeddings": 1024,
  "model_type": "mbart",
  "normalize_before": true

In [7]:
# Check what token 250181 is
print(tokenizer.decode([250181]))  # should be something like "en_XX" or an AMR token
print(tokenizer.decode([250020]))  # the standard en_XX token for comparison

# Check lang codes
print(tokenizer.lang_code_to_id)

<AMR>
ro_RO
{'ar_AR': 250001, 'cs_CZ': 250002, 'de_DE': 250003, 'en_XX': 250004, 'es_XX': 250005, 'et_EE': 250006, 'fi_FI': 250007, 'fr_XX': 250008, 'gu_IN': 250009, 'hi_IN': 250010, 'it_IT': 250011, 'ja_XX': 250012, 'kk_KZ': 250013, 'ko_KR': 250014, 'lt_LT': 250015, 'lv_LV': 250016, 'my_MM': 250017, 'ne_NP': 250018, 'nl_XX': 250019, 'ro_RO': 250020, 'ru_RU': 250021, 'si_LK': 250022, 'tr_TR': 250023, 'vi_VN': 250024, 'zh_CN': 250025}


In [10]:
text = "This is a simple test sentence."

tokens = tokenizer.tokenize(text)
print(tokens)
encoded = tokenizer(text)
print(encoded["input_ids"])
encoding = tokenizer(text)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
ids = encoding["input_ids"]

for token, idx in zip(tokens, ids):
    print(f"{token:15} -> {idx}")

['▁This', '▁is', '▁a', '▁simple', '▁test', '▁sentence', '.']
[3293, 83, 10, 8781, 3034, 149357, 5, 2, 250004]
▁This           -> 3293
▁is             -> 83
▁a              -> 10
▁simple         -> 8781
▁test           -> 3034
▁sentence       -> 149357
.               -> 5
</s>            -> 2
en_XX           -> 250004


In [8]:
def text_to_amr(sentence: str):
    tokenizer.src_lang = "en_XX"
    
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=512,
            num_beams=5,
            early_stopping=True,
            forced_bos_token_id=250181,  # <AMR> token
        )

    # Skip special tokens but keep AMR structure
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [29]:
text = "These rates have been increased across the entire country."
amr_text = text_to_amr(text)
print(amr_text)

Both `max_new_tokens` (=512) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<AMR><AMR><rel><pointer:0>increase-01:ARG1<rel><pointer:1> rate:mod<rel><pointer:2> this</rel></rel>:location<rel><pointer:3> across:op1<rel><pointer:4> country:mod<rel><pointer:5> entire</rel></rel></rel></rel>


In [9]:
text = "Currently, the prices of CNG, PNG gas, and edible oil have increased."
amr_text = text_to_amr(text)
print(amr_text)

Both `max_new_tokens` (=512) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<AMR><AMR><rel><pointer:0>increase-01:ARG1<rel><pointer:1>monetary-quantity:ARG2-of<rel><pointer:2>price-01:ARG1<rel><pointer:3> and:op1<rel><pointer:4> gas:mod<rel><pointer:5> small-molecule:wiki<lit> Carbon dioxide</lit>:name<rel><pointer:6> name:op1<lit> CNG</lit></rel></rel></rel>:op2<rel><pointer:7> gas:mod<rel><pointer:8> petroleum</rel></rel>:op3<rel><pointer:9> oil:ARG1-of<rel><pointer:10>eat-01:ARG1-of<rel><pointer:11>possible-01</rel></rel></rel></rel></rel></rel>:time<rel><pointer:12> current</rel></rel>
